![Foundations of NLP] Count Vectorizer and TF-IDF()

---



### About:  
Natural Language Processing (NLP) is the field of computer science and artificial intelligence that uses principles of statistics and probability to translate human language into a form that computers can use and apply to solve problems. In this notebook we will convert our text into features that can be used to predict star rating of a new product. 

### Objective:
1. Describe the rationale and methods for representing text numerically 
2. Create a frequency word vector with CountVectorizer 
3. Explore the impact of incorporating TF-IDF 
4. Apply CountVectorizer and TF-IDF to predict customer rating based on customer review 


### Prerequisites:
tktk




In [96]:
# imports
import numpy as np
import pandas as pd

# NLP imports
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# model building imports
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

### NLP Scenario 
You are an analyst for a marketing company that just launched a new product suite of mobile devices. You have data from product reviews of the new product. We need to draft a report for our client that reports the sentiment of the reviews. We will start by trying to predict the review rating using the text data as features. 


#####  Product Reviews 
1. "I absolutely love the TechWave X1! It has made my daily tasks so much easier and more efficient. Highly recommend it!"
2. "I'm not very impressed with the TechWave X1. It lacks some essential features and is quite slow."
3. "The TechWave X1 is fantastic! It has exceeded my expectations and has become an essential part of my daily routine."
4. "I found the TechWave X1 to be quite average. It does the job, but there's nothing particularly special about it."
5. "The TechWave X1 is terrible. It's full of glitches and crashes frequently. I regret purchasing it."
6.  "The TechWave X1 is disappointing. It doesn't live up to the hype and is missing several key functionalities."

### N-grams Introduction 
Computers understand the world in numbers. In order to use text data we have to first represent it in a way that computers can interpret. 

Let's convert the text above to a data frame and start working with the text data. 

In [98]:
# data setup
reviews_with_ratings = [
    ("I absolutely love the TechWave X1! It has made my daily tasks so much easier and more efficient. Highly recommend it!", 5),
    ("I'm not very impressed with the TechWave X1. It lacks some essential features and is quite slow.", 2),
    ("The TechWave X1 is fantastic! It has exceeded my expectations and has become an essential part of my daily routine.", 5),
    ("I found the TechWave X1 to be quite average. It does the job, but there's nothing particularly special about it.", 3),
    ("The TechWave X1 is terrible. It's full of glitches and crashes frequently. I regret purchasing it.", 1),
    ("The TechWave X1 is disappointing. It doesn't live up to the hype and is missing several key functionalities.", 2)
]

# Create a DataFrame
df = pd.DataFrame(reviews_with_ratings, columns=['reviews', 'star_rating'])
df.head()

,reviews,star_rating
0,I absolutely love the TechWave X1! It has made...,5
1,I'm not very impressed with the TechWave X1. I...,2
2,The TechWave X1 is fantastic! It has exceeded ...,5
3,I found the TechWave X1 to be quite average. I...,3
4,The TechWave X1 is terrible. It's full of glit...,1


Below we will convert the text from the reviews into a matrix that creates a column for each word and tracks which reviews contain each word. Take a look at the output below. 

In [74]:
# Initialize CountVectorizer for n-grams (single words in this case)
vectorizer = CountVectorizer(ngram_range=(1, 1))

# Fit and transform the reviews
X = vectorizer.fit_transform(df['reviews'])

# Create a new DataFrame with the n-grams
ngrams_df = pd.DataFrame(
    X.toarray(), columns=vectorizer.get_feature_names_out())
print("ngrams_df shape:", ngrams_df.shape)
print("ngrams_df columns:", ngrams_df.columns)

# Display the new DataFrame
ngrams_df.head()

ngrams_df shape: (6, 66)
ngrams_df columns: Index(['about', 'absolutely', 'an', 'and', 'average', 'be', 'become', 'but',
       'crashes', 'daily', 'disappointing', 'does', 'doesn', 'easier',
       'efficient', 'essential', 'exceeded', 'expectations', 'fantastic',
       'features', 'found', 'frequently', 'full', 'functionalities',
       'glitches', 'has', 'highly', 'hype', 'impressed', 'is', 'it', 'job',
       'key', 'lacks', 'live', 'love', 'made', 'missing', 'more', 'much', 'my',
       'not', 'nothing', 'of', 'part', 'particularly', 'purchasing', 'quite',
       'recommend', 'regret', 'routine', 'several', 'slow', 'so', 'some',
       'special', 'tasks', 'techwave', 'terrible', 'the', 'there', 'to', 'up',
       'very', 'with', 'x1'],
      dtype='object')


,about,absolutely,an,and,average,be,become,but,crashes,daily,...,tasks,techwave,terrible,the,there,to,up,very,with,x1
0,0,1,0,1,0,0,0,0,0,1,...,1,1,0,1,0,0,0,0,0,1
1,0,0,0,1,0,0,0,0,0,0,...,0,1,0,1,0,0,0,1,1,1
2,0,0,1,1,0,0,1,0,0,1,...,0,1,0,1,0,0,0,0,0,1
3,1,0,0,0,1,1,0,1,0,0,...,0,1,0,2,1,1,0,0,0,1
4,0,0,0,1,0,0,0,0,1,0,...,0,1,1,1,0,0,0,0,0,1


### Stop words
Notice that we have some words that are so common (for example: "an", "and", "the") that they are not useful for our analysis. These words are called stop words. We can remove them by using the stop_words parameter in the CountVectorizer class. Notice the difference in the output below. 

In [75]:
# Initialize CountVectorizer for n-grams (single words in this case)
vectorizer = CountVectorizer(ngram_range=(1, 1), stop_words='english')

# Fit and transform the reviews
X = vectorizer.fit_transform(df['reviews'])

# Create a new DataFrame with the n-grams
ngrams_df = pd.DataFrame(
    X.toarray(), columns=vectorizer.get_feature_names_out())
print("ngrams_df shape:", ngrams_df.shape)
print("ngrams_df columns:", ngrams_df.columns)

# Display the new DataFrame
ngrams_df.head()

ngrams_df shape: (6, 38)
ngrams_df columns: Index(['absolutely', 'average', 'crashes', 'daily', 'disappointing', 'does',
       'doesn', 'easier', 'efficient', 'essential', 'exceeded', 'expectations',
       'fantastic', 'features', 'frequently', 'functionalities', 'glitches',
       'highly', 'hype', 'impressed', 'job', 'key', 'lacks', 'live', 'love',
       'missing', 'particularly', 'purchasing', 'quite', 'recommend', 'regret',
       'routine', 'slow', 'special', 'tasks', 'techwave', 'terrible', 'x1'],
      dtype='object')


,absolutely,average,crashes,daily,disappointing,does,doesn,easier,efficient,essential,...,quite,recommend,regret,routine,slow,special,tasks,techwave,terrible,x1
0,1,0,0,1,0,0,0,1,1,0,...,0,1,0,0,0,0,1,1,0,1
1,0,0,0,0,0,0,0,0,0,1,...,1,0,0,0,1,0,0,1,0,1
2,0,0,0,1,0,0,0,0,0,1,...,0,0,0,1,0,0,0,1,0,1
3,0,1,0,0,0,1,0,0,0,0,...,1,0,0,0,0,1,0,1,0,1
4,0,0,1,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,1,1,1


We went from 58 n-grams/words to 34 n-grams/words by removing the stop words. This will allow for more meaningful analysis of the reviews.  

### N-gram Length
Another modification we may want to make is adjusting how words are grouped. By using single words, we are missing important context such as "Techwave X1" being the product name of interest. Let's edit our code to capture bigrams. 

In [76]:
# Initialize CountVectorizer for n-grams (single words in this case)
vectorizer = CountVectorizer(ngram_range=(2, 2), stop_words='english')

# Fit and transform the reviews
X = vectorizer.fit_transform(df['reviews'])

# Create a new DataFrame with the n-grams
ngrams_df = pd.DataFrame(
    X.toarray(), columns=vectorizer.get_feature_names_out())
print("ngrams_df shape:", ngrams_df.shape)
print("ngrams_df columns:", ngrams_df.columns)

# Display the new DataFrame
ngrams_df.head()

ngrams_df shape: (6, 40)
ngrams_df columns: Index(['absolutely love', 'average does', 'crashes frequently',
       'daily routine', 'daily tasks', 'disappointing doesn', 'does job',
       'doesn live', 'easier efficient', 'efficient highly', 'essential daily',
       'essential features', 'exceeded expectations', 'expectations essential',
       'fantastic exceeded', 'features quite', 'frequently regret',
       'glitches crashes', 'highly recommend', 'hype missing',
       'impressed techwave', 'job particularly', 'key functionalities',
       'lacks essential', 'live hype', 'love techwave', 'missing key',
       'particularly special', 'quite average', 'quite slow',
       'regret purchasing', 'tasks easier', 'techwave x1', 'terrible glitches',
       'x1 daily', 'x1 disappointing', 'x1 fantastic', 'x1 lacks', 'x1 quite',
       'x1 terrible'],
      dtype='object')


,absolutely love,average does,crashes frequently,daily routine,daily tasks,disappointing doesn,does job,doesn live,easier efficient,efficient highly,...,regret purchasing,tasks easier,techwave x1,terrible glitches,x1 daily,x1 disappointing,x1 fantastic,x1 lacks,x1 quite,x1 terrible
0,1,0,0,0,1,0,0,0,1,1,...,0,1,1,0,1,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,1,0,0,0,0,1,0,0
2,0,0,0,1,0,0,0,0,0,0,...,0,0,1,0,0,0,1,0,0,0
3,0,1,0,0,0,0,1,0,0,0,...,0,0,1,0,0,0,0,0,1,0
4,0,0,1,0,0,0,0,0,0,0,...,1,0,1,1,0,0,0,0,0,1


Interesting! Let's edit this a bit more to capture both bigrams and single words. 

In [77]:
# Initialize CountVectorizer for n-grams (single words in this case)
vectorizer = CountVectorizer(ngram_range=(1, 2), stop_words='english')

# Fit and transform the reviews
X = vectorizer.fit_transform(df['reviews'])

# Create a new DataFrame with the n-grams
ngrams_df = pd.DataFrame(
    X.toarray(), columns=vectorizer.get_feature_names_out())
print("ngrams_df shape:", ngrams_df.shape)
print("ngrams_df columns:", ngrams_df.columns)

# Display the new DataFrame
ngrams_df.head()

ngrams_df shape: (6, 78)
ngrams_df columns: Index(['absolutely', 'absolutely love', 'average', 'average does', 'crashes',
       'crashes frequently', 'daily', 'daily routine', 'daily tasks',
       'disappointing', 'disappointing doesn', 'does', 'does job', 'doesn',
       'doesn live', 'easier', 'easier efficient', 'efficient',
       'efficient highly', 'essential', 'essential daily',
       'essential features', 'exceeded', 'exceeded expectations',
       'expectations', 'expectations essential', 'fantastic',
       'fantastic exceeded', 'features', 'features quite', 'frequently',
       'frequently regret', 'functionalities', 'glitches', 'glitches crashes',
       'highly', 'highly recommend', 'hype', 'hype missing', 'impressed',
       'impressed techwave', 'job', 'job particularly', 'key',
       'key functionalities', 'lacks', 'lacks essential', 'live', 'live hype',
       'love', 'love techwave', 'missing', 'missing key', 'particularly',
       'particularly special', 'purchas

,absolutely,absolutely love,average,average does,crashes,crashes frequently,daily,daily routine,daily tasks,disappointing,...,techwave x1,terrible,terrible glitches,x1,x1 daily,x1 disappointing,x1 fantastic,x1 lacks,x1 quite,x1 terrible
0,1,1,0,0,0,0,1,0,1,0,...,1,0,0,1,1,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,1,0,0,1,0,0,0,1,0,0
2,0,0,0,0,0,0,1,1,0,0,...,1,0,0,1,0,0,1,0,0,0
3,0,0,1,1,0,0,0,0,0,0,...,1,0,0,1,0,0,0,0,1,0
4,0,0,0,0,1,1,0,0,0,0,...,1,1,1,1,0,0,0,0,0,1


Great job! You've successfully created n-grams from the reviews using CountVectorizer. You've also learned how to filter out stop words and create n-grams of different lengths. This will be useful when you start building machine learning models to analyze text data. 

### TF-IDF

Now let's explore another popular technique for text analysis: TF-IDF (Term Frequency-Inverse Document Frequency). TF-IDF is a numerical statistic that reflects how important a word is to a document in a collection or corpus. It is often used in information retrieval and text mining. Let's see how it works in practice.

In [78]:
# Initialize TfidfVectorizer
vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words='english')

# Fit and transform the reviews
X = vectorizer.fit_transform(df['reviews'])

# Create a new DataFrame with the TF-IDF values
tfidf_df = pd.DataFrame(
    X.toarray(), columns=vectorizer.get_feature_names_out())
print("tfidf_df shape:", tfidf_df.shape)
print("tfidf_df columns:", tfidf_df.columns)

# Display the new DataFrame
tfidf_df.head()

tfidf_df shape: (6, 78)
tfidf_df columns: Index(['absolutely', 'absolutely love', 'average', 'average does', 'crashes',
       'crashes frequently', 'daily', 'daily routine', 'daily tasks',
       'disappointing', 'disappointing doesn', 'does', 'does job', 'doesn',
       'doesn live', 'easier', 'easier efficient', 'efficient',
       'efficient highly', 'essential', 'essential daily',
       'essential features', 'exceeded', 'exceeded expectations',
       'expectations', 'expectations essential', 'fantastic',
       'fantastic exceeded', 'features', 'features quite', 'frequently',
       'frequently regret', 'functionalities', 'glitches', 'glitches crashes',
       'highly', 'highly recommend', 'hype', 'hype missing', 'impressed',
       'impressed techwave', 'job', 'job particularly', 'key',
       'key functionalities', 'lacks', 'lacks essential', 'live', 'live hype',
       'love', 'love techwave', 'missing', 'missing key', 'particularly',
       'particularly special', 'purchasin

,absolutely,absolutely love,average,average does,crashes,crashes frequently,daily,daily routine,daily tasks,disappointing,...,techwave x1,terrible,terrible glitches,x1,x1 daily,x1 disappointing,x1 fantastic,x1 lacks,x1 quite,x1 terrible
0,0.247966,0.247966,0.000000,0.000000,0.000000,0.000000,0.203336,0.000000,0.247966,0.0,...,0.110072,0.000000,0.000000,0.110072,0.247966,0.0,0.000000,0.000000,0.000000,0.000000
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.128486,0.000000,0.000000,0.128486,0.000000,0.0,0.000000,0.289448,0.000000,0.000000
2,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.237352,0.289448,0.000000,0.0,...,0.128486,0.000000,0.000000,0.128486,0.000000,0.0,0.289448,0.000000,0.000000,0.000000
3,0.000000,0.000000,0.285556,0.285556,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,...,0.126758,0.000000,0.000000,0.126758,0.000000,0.0,0.000000,0.000000,0.285556,0.000000
4,0.000000,0.000000,0.000000,0.000000,0.281817,0.281817,0.000000,0.000000,0.000000,0.0,...,0.125098,0.281817,0.281817,0.125098,0.000000,0.0,0.000000,0.000000,0.000000,0.281817


Notice that the TF-IDF values are different from the count values. This is because the TF-IDF values are normalized by the term frequency and inverse document frequency, which gives more weight to terms that are rare in the corpus and less weight to terms that are common. This helps to identify the most important terms in a document.

### Lab 
Use count vectorizer and/or TF-IDF to predict the start rating of the reviews for the TechWave X1 




In [81]:
df_x1 = pd.read_csv('./assets/reviews.csv')
df_x1['rating'] = pd.to_numeric(df_x1['rating'])
df_x1.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 220 entries, 0 to 219
Data columns (total 4 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   date     220 non-null    object
 1   review   220 non-null    object
 2   rating   220 non-null    int64 
 3   product  220 non-null    object
dtypes: int64(1), object(3)
memory usage: 7.0+ KB


In [84]:
# Convert 5-star ratings to binary sentiment
# 1-3: Negative (0), 4-5: Positive (1)
df_x1['sentiment'] = df_x1['rating'].apply(lambda x: 0 if x <= 3 else 1)
pd.crosstab(df_x1['rating'], df_x1['sentiment'])

sentiment,0,1
rating,,
1,5,0
2,20,0
3,14,0
4,0,37
5,0,144


In [85]:
df_x1.head()

,date,review,rating,product,sentiment
0,2024-01-01,Absolutely love the 48MP main camera and HDR s...,5,TechWave X1,1
1,2024-01-01,Major disappointment. Constant heating issues ...,2,TechWave X1,0
2,2024-01-02,The optical stabilization and 4K video recordi...,3,TechWave X1,0
3,2024-01-02,Cannot recommend this device. Poor RAM managem...,1,TechWave X1,0
4,2024-01-03,Outstanding performance for daily tasks. The 5...,5,TechWave X1,1


In [86]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    df_x1['review'], df_x1['sentiment'], test_size=0.2, random_state=1212)

In [97]:
# Initialize a CountVectorizer and predict star ratings


# Initialize a CountVectorizer or TfidfVectorizer
vectorizer = CountVectorizer(ngram_range=(1, 2), stop_words='english')

# Fit and transform the training data
X_train_transformed = vectorizer.fit_transform(X_train)

# Transform the testing data
X_test_transformed = vectorizer.transform(X_test)

# Initialize a Logistic Regression model
model = LogisticRegression()

# Train the model on the transformed training data
model.fit(X_train_transformed, y_train)

# Predict the ratings for the testing data
y_pred = model.predict(X_test_transformed)

# Evaluate the model's performance
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

# Display the confusion matrix
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.8181818181818182
[[ 2  8]
 [ 0 34]]


In [94]:
# Initialize TfidfVectorizer
vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words='english')

# Fit and transform the training data
X_train_transformed = vectorizer.fit_transform(X_train)

# Transform the testing data
X_test_transformed = vectorizer.transform(X_test)

# Initialize a Logistic Regression model
model = LogisticRegression()

# Train the model on the transformed training data
model.fit(X_train_transformed, y_train)

# Predict the ratings for the testing data
y_pred = model.predict(X_test_transformed)

# Evaluate the model's performance
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

# Display the confusion matrix
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.7727272727272727
[[ 0 10]
 [ 0 34]]


### Summary 
In this lab we explored one way to represent text numerically so that it can be used for by the computer for analysis. This is surprisingly effective, but strips the context from the words. The context, or the relationship between words, can be extremely important for interpreting meaning. Other embedding methods (such as word2vec) have been created to allow us to track context. 